# Reto 2 — Visión: identificación, ángulo y recuperación en COIL-100

Notebook de punta a punta y reproducible: **adquisición de datos → EDA → exploración y selección de modelo →
evaluación de la receta productiva congelada → análisis de errores → costo → exportación del artefacto**. Toda la lógica vive en `src/vision/`;
aquí solo se orquesta y se muestran resultados. Semilla global 42. Los experimentos quedan registrados en
MLflow (`sqlite:///mlflow.db`, experimento `vision-coil100`).

**Regla metodológica:** todo hiperparámetro y todo umbral se elige con particiones de *validación* construidas
exclusivamente con las vistas de entrenamiento de cada protocolo; ningún número de test se usa para elegir nada.
Hay dos niveles: (i) **selección anidada por protocolo** (sección 4): familia de clasificador, hiperparámetros,
k angular y umbrales se eligen solo con la validación interna del train de ese protocolo y se evalúa una vez en
su test — son las únicas cifras que presentamos como estimación independiente; (ii) la **configuración
desplegada** (sección 5, receta congelada en `src/vision/recipe.py`), cuya familia/C se eligió con validaciones
agregadas de los cuatro protocolos que, en conjunto, se solapan con tests de otros protocolos: sus cifras se
reportan con esa salvedad.

In [1]:
import sys, json, time, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.vision import data as vdata, analysis as va, experiments as vexp
from src.vision.angle import circular_error
from src.vision.data import ensure_dataset, sha256_file, data_version_hash, protocol_split, validation_split, COIL_SHA256
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
np.random.seed(42)
T0 = time.time()

## 1. Adquisición y verificación de datos

Si `data/raw/coil-100/` no existe, se descarga el zip oficial
(`https://www.cs.columbia.edu/CAVE/databases/SLAM_coil-20_coil-100/coil-100/coil-100.zip`), se verifica su
SHA-256 y se extrae. Además se calcula un **hash de versión de datos** (SHA-256 sobre nombre + contenido de
cada PNG) que viaja en MLflow y en `metadata.json` del artefacto.

In [2]:
img_dir = ensure_dataset()
zip_path = img_dir.parent / "coil-100.zip"
if zip_path.exists():
    print("SHA-256 zip:", sha256_file(zip_path), "OK" if sha256_file(zip_path) == COIL_SHA256 else "DISTINTO")
DATA_HASH = data_version_hash()
print("imágenes:", len(list(img_dir.glob("obj*__*.png"))), "| hash de versión de datos:", DATA_HASH[:16], "…")
data = vexp.load_all()          # imágenes + características clásicas + píxeles crudos + embeddings ResNet18 (cache en data/runtime)
ds = data["ds"]
print(ds.images.shape, "clases:", len(np.unique(ds.labels)), "ángulos:", np.unique(ds.angles)[:4], "…", np.unique(ds.angles)[-1])

SHA-256 zip: fcb4b8b01c0778f416d8d7a09f45082c4f7075cc74743dcddd5c44307bd4afe7 OK


imágenes: 7200 | hash de versión de datos: e9e73124fbf0e964 …


(7200, 128, 128, 3) clases: 100 ángulos: [ 0  5 10 15] … 355


## 2. Análisis exploratorio (EDA)

In [3]:
fig = va.sample_grid(ds); va.savefig(fig, "01_sample_grid"); plt.show()
st = va.image_stats(ds)
desc = pd.DataFrame({k: pd.Series(v).describe(percentiles=[.001, .01, .5, .99, .999]) for k, v in st.items()}).T
display(desc.round(3))

,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
brightness,7200.0,0.268,0.071,0.129,0.134,0.151,0.265,0.440,0.491,0.526
object_fraction,7200.0,0.471,0.147,0.128,0.131,0.185,0.468,0.781,0.930,0.952
background_median,7200.0,26.297,2.154,22.000,23.000,24.000,26.000,35.000,43.000,47.000
sharpness,7200.0,327.561,251.640,44.245,46.601,54.948,255.630,1282.385,1810.964,2041.270


**Lectura.** 100 objetos × 72 vistas (paso exacto de 5°), 128×128 RGB, **fondo oscuro uniforme** (mediana del
borde ≈ 25/255, *no* 0) e **iluminación controlada**: brillo y nitidez varían poco. Consecuencias de diseño:

* El problema de clase es "fácil" en laboratorio; el riesgo real está en el ángulo y en la extrapolación a planta.
* Como el fondo es casi constante, una máscara por umbral de gris (> 40) separa objeto/fondo; en planta esto no
  vale (cinta, reflejos) → compuerta de calidad + segmentación robusta serían requisitos.
* Las estadísticas de brillo/nitidez/fracción de objeto definen los umbrales de la **compuerta de calidad**
  determinista (`src/vision/quality.py`, percentil extremo × margen).

In [4]:
from src.vision.models import VisionModel
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
Zall = PCA(150, random_state=42).fit_transform(StandardScaler().fit_transform(data["classic"]))  # solo para EDA
sym = va.symmetry_analysis(Zall, ds)
fig = va.symmetry_figure(sym); va.savefig(fig, "02_symmetry"); plt.show()
symdf = pd.DataFrame({"perfil": sym["profile_score"], "loo": sym["loo_score"], "d180/d90": sym["ratio_180_90"]})
symdf.index = [f"obj{o}" for o in symdf.index]
print("Más rotacionalmente ambiguos según el perfil (d@5° / d@≥45°):")
display(symdf.sort_values("perfil", ascending=False).head(10).round(3))
print("Vista 'gemela' más parecida a ≥30° (índice LOO):")
display(symdf.sort_values("loo", ascending=False).head(8).round(3))
print(f"Objetos con vistas opuestas (180°) más parecidas que a 90°: {(symdf['d180/d90'] < 1).sum()} de 100")

Más rotacionalmente ambiguos según el perfil (d@5° / d@≥45°):


,perfil,loo,d180/d90
obj95,0.710,0.042,1.060
obj94,0.612,0.111,1.079
obj47,0.517,0.069,1.171
obj25,0.513,0.000,0.970
obj34,0.505,0.000,1.082
obj50,0.496,0.000,0.890
obj30,0.485,0.000,0.745
obj58,0.459,0.000,0.694
obj86,0.453,0.000,1.001
obj56,0.444,0.000,1.151


Vista 'gemela' más parecida a ≥30° (índice LOO):


,perfil,loo,d180/d90
obj98,0.336,0.500,0.255
obj84,0.328,0.472,0.253
obj67,0.336,0.431,0.249
obj1,0.355,0.236,0.379
obj12,0.300,0.208,0.316
obj96,0.372,0.208,0.446
obj51,0.273,0.167,0.311
obj79,0.353,0.111,0.459


Objetos con vistas opuestas (180°) más parecidas que a 90°: 72 de 100


**Simetría rotacional (cuantificada).** Para cada objeto se mide la distancia media en el espacio de
características entre pares de vistas en función de su separación angular. En un objeto asimétrico la curva
crece con la separación; en uno simétrico (latas, tazas, donas, frascos: p. ej. obj95, obj94, obj47, obj25)
es casi plana → **el ángulo es intrínsecamente ambiguo**. El índice LOO (¿la vista más parecida está a ≥30°?)
detecta además objetos con vistas a 180° casi idénticas (cajas/frascos con dos caras parecidas).

**Tratamiento:** (i) a nivel de instancia, `estimate_angle` marca `ambiguous` si existe una vista de referencia
a ≥30° del ángulo estimado con distancia ≤ 1.10 × la mejor; (ii) a nivel de clase, el modelo productivo
(referencias cada 5°) marca como simétricas las clases con índice LOO ≥ 0.25; (iii) **empíricamente**: en
cada protocolo, las clases cuyo ángulo falla (> ±10°) en más del 25 % de sus vistas de *validación* se marcan
como de ángulo poco fiable (detecta las donas/tazas obj47, obj94, obj95 sin mirar el test); (iv) todo esto genera
`rotational_ambiguity` → **REVIEW** (el robot recibe HOLD).

**Vistas a 180°:** en ~3/4 de los objetos la vista opuesta (180°) está más cerca en el espacio de
características que la vista a 90° (la silueta de un objeto visto desde lados opuestos es casi la imagen espejo).
Por eso el ángulo no se estima con una media circular de todos los vecinos (se cancelarían modos opuestos), sino
con el modo del vecino más cercano refinado localmente, y la ambigüedad se marca explícitamente. Alternativa documentada: para un objeto
perfectamente simétrico el ángulo no importa para el agarre y podría reportarse "módulo simetría"; no lo
hacemos por defecto porque la etiqueta/código de barras sí puede depender del ángulo.

In [5]:
pairs = va.confusable_pairs(Zall, ds, top=10)
display(pd.DataFrame(pairs).round(2))

,a,b,min_inter_distance,mean_intra_nn_distance
0,obj61,obj92,10.87,7.41
1,obj29,obj32,16.89,8.27
2,obj67,obj84,17.99,13.39
3,obj47,obj94,19.32,3.98
4,obj15,obj91,19.40,14.02
5,obj21,obj44,19.66,15.64
6,obj15,obj69,19.77,14.42
7,obj84,obj98,19.78,13.29
8,obj62,obj93,20.56,9.20
9,obj23,obj69,20.94,15.01


**Clases casi duplicadas.** Los pares con menor distancia inter-objeto (comparada con la distancia típica entre
vistas adyacentes del mismo objeto) son los candidatos a confusión: mismas formas con distinto color/texto
(p. ej. cajas o latas de la misma familia). Esto motiva incluir **color** (HSV) además de forma (HOG).

**Laboratorio vs. planta.** COIL-100 tiene fondo negro, luz controlada, cámara fija a la misma distancia,
un único eje de rotación (tornamesa), sin oclusión ni desenfoque por movimiento. En planta habrá: cinta con
textura y reflejos, iluminación variable, oclusión parcial/objetos múltiples, movimiento de la cinta (blur),
pose de cámara y escala variables, empaques nuevos. Mitigaciones: (1) aumentación de datos (brillo/contraste,
blur, ruido, escala, recortes) y captura de vistas reales por SKU en la estación; (2) compuerta de calidad
(oscuridad, saturación, nitidez, cuadro vacío) antes del modelo; (3) re-calibrar umbrales con datos de planta
(los umbrales son configurables en caliente vía `/v1/admin/thresholds`); (4) **piloto en sombra** con revisión
humana del 100% antes de habilitar ACT, y monitoreo de la tasa ACT/REVIEW/REJECT y de la deriva de distancias.

## 3. Protocolos y esquema de validación

| Protocolo | Train (por objeto) | Test | Validación (solo dentro de train) |
|---|---|---|---|
| A Interpolación | 0,10,…,350 (36) | 5,15,…,355 (36) | fit paso 20°, val el resto (brecha 10° > 5° del test) |
| B Sector ciego | 60 vistas | arco contiguo de 60° (12), inicio aleatorio por objeto (semilla 42) | se oculta otro arco de 60° dentro de train (semilla 43) |
| C Escasez | 1 cada 30° (12) | las otras 60 | fit paso 60°, val el resto |
| D estricto | 90 objetos, **todas** sus vistas (72) | solo las 720 vistas de los 10 objetos desconocidos | 10 pseudo-desconocidos entre los 90 (semilla 43); fit paso 10° de los otros 80 |
| D variante | 90 objetos, paso 10° (36) | vistas impares de los 90 + todas las de los 10 desconocidos | 10 pseudo-desconocidos entre los 90 (semilla 43) |

**D estricto** es el protocolo pedido: como el test solo tiene desconocidos, no se puede calcular AUROC; se
reporta la tasa de desconocidos rechazados y la de **desconocidos que llegan a ACT** (falsa aceptación). La
**variante D** (experimento adicional) entrena con la mitad de las vistas de los conocidos para poder medir
*ambos lados* con vistas no vistas (AUROC, conocidos rechazados por error). La validación imita (y suele exagerar)
la brecha de cada protocolo; en C la brecha de validación (30°) es mayor que la del test (≤15°).

In [6]:
rows = []
for p in ("A", "B", "C", "D_strict", "D"):
    sp = protocol_split(ds.labels, ds.angles, p); vs = validation_split(ds.labels, ds.angles, p)
    rows.append({"protocolo": p, "train": len(sp["train"]), "test": len(sp["test"]), "fit (val)": len(vs["fit"]),
                 "val": len(vs["val"]), "fuga train∩test": len(np.intersect1d(sp["train"], sp["test"])),
                 "val ⊂ train": bool(np.isin(np.r_[vs["fit"], vs["val"]], sp["train"]).all())})
display(pd.DataFrame(rows))
spB = protocol_split(ds.labels, ds.angles, "B"); spD = protocol_split(ds.labels, ds.angles, "D")
print("B: inicio del arco ciego (primeros 10 objetos):", {f"obj{k}": v for k, v in list(spB["arc_starts"].items())[:10]})
print("D: objetos desconocidos (semilla 42):", [f"obj{o}" for o in spD["unknown_objects"]])

,protocolo,train,test,fit (val),val,fuga train∩test,val ⊂ train
0,A,3600,3600,1800,1800,0,True
1,B,6000,1200,4800,1200,0,True
2,C,1200,6000,600,600,0,True
3,D_strict,6480,720,2880,3600,0,True
4,D,3240,3960,1440,1800,0,True


B: inicio del arco ciego (primeros 10 objetos): {'obj1': 30, 'obj2': 275, 'obj3': 235, 'obj4': 155, 'obj5': 155, 'obj6': 305, 'obj7': 30, 'obj8': 250, 'obj9': 70, 'obj10': 30}
D: objetos desconocidos (semilla 42): ['obj9', 'obj10', 'obj20', 'obj42', 'obj61', 'obj69', 'obj72', 'obj83', 'obj95', 'obj97']


## 4. Selección anidada por protocolo (cifras independientes)

**Preprocesado** (`features.preprocess`, igual en entrenamiento e inferencia): mediana 3×3, máscara de objeto
sobre el gris suavizado (Gauss 5×5, > 40) con apertura 3×3, y fondo puesto a 0. Motivo: en la prueba de punta a
punta, ruido gaussiano de sensor casi imperceptible (σ=1–4) sobre el fondo creaba textura que HOG/LBP/histograma
captaban y disparaba la distancia de novedad (REJECT por "desconocido").

**Aumentación** (solo vistas de entrenamiento): 1 copia por vista con ruido σ~U(0,6), brillo ×U(0.8,1.2) y blur
leve (p=0.3). La **validación** usa sus vistas limpias + 1 copia perturbada con la misma distribución, para que
los umbrales reflejen condiciones realistas. Las vistas de test nunca se perturban para entrenar ni validar.

**Características:** histograma HSV 16×4×4 (Hellinger) sobre la máscara, HOG 64×64, LBP uniforme 2×2 y Hu;
estandarización + PCA(150) ajustados solo con train.

**Candidatos** (6): logística C∈{0.1,1,10}, SVM RBF C=10 (Platt), kNN k=1, Random Forest 300. Para cada uno, en
la validación interna del protocolo: k angular ∈{1,2,3}, clases de ángulo poco fiable, y umbrales. Se elige el
candidato con automatización habilitada y mayor cobertura de validación (desempate: exactitud de validación).

**Ángulo e incertidumbre:** recuperación condicionada a la clase predicha; el *score* de incertidumbre en grados
traduce la distancia al vecino más cercano en "grados de giro equivalentes" con la curva distancia-vs-separación
del objeto. **No es una cota del error**; su umbral se elige por el riesgo empírico.

**Reglas para habilitar ACT (en validación):** error entre ACT ≤ 2 % con ≥ 50 ACT; y, si hay pseudo-desconocidos
(≥ 100 vistas), que como mucho el 5 % de ellos llegue a ACT. Si no se cumple → automatización desactivada:
política efectiva 0 ACT, riesgo `null` (las cifras del candidato descartado se guardan aparte).

In [7]:
from src.vision import recipe as vrec
OUT_DIR = ROOT / "data" / "runtime" / "experiments"
t = time.time()
R = vrec.run_recipe_all(data, features=("classic", "resnet18"), out_dir=OUT_DIR)
print(f"selección anidada + configuración desplegada + robustez: {time.time() - t:.0f}s")
RS = va.build_recipe_summary(R["results"], R["baselines"], R["nested"])
def fmt_ci(x):
    return "N/A" if x is None or x[0] is None else f"[{x[0]:.2%}, {x[1]:.2%}]"
def rowfmt(p, lab, m):
    return {"protocolo": p, "modelo": lab, "exactitud": m.get("class_accuracy"), "MAE°": m.get("angle_mae_deg"),
            "éxito e2e": m.get("end_to_end_success"), "% ACT": m["coverage_act"], "n ACT": m.get("n_act"),
            "error|ACT": "N/A" if m["error_among_act"] is None else round(m["error_among_act"], 4),
            "IC95": fmt_ci(m.get("error_among_act_wilson95")), "automatización": m.get("automation_enabled", "—")}
nrows = [{**rowfmt(p, RS[p]["nested_classic"]["selected_candidate"], RS[p]["nested_classic"]), "k": RS[p]["nested_classic"]["k"]} for p in RS]
print("CIFRAS INDEPENDIENTES (selección anidada, test evaluado una vez):"); display(pd.DataFrame(nrows))
for p in RS:
    print(f"Validación interna del protocolo {p} (candidatos):")
    display(pd.DataFrame(R["nested"][p]["selection_table_val"]).round(4))

2026/10/24 13:19:36 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/nico/Documents/IA/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


selección anidada + configuración desplegada + robustez: 289s
CIFRAS INDEPENDIENTES (selección anidada, test evaluado una vez):


,protocolo,modelo,exactitud,MAE°,éxito e2e,% ACT,n ACT,error|ACT,IC95,automatización,k
0,A,svm_rbf:{'C': 10.0},1.000000,2.911346,0.985000,0.903056,3251,0.0009,"[0.03%, 0.27%]",True,2
1,B,logreg:{'C': 1.0},0.961667,65.515598,0.285000,0.000000,0,N/A,N/A,False,1
2,C,logreg:{'C': 0.1},0.987833,13.006580,0.697167,0.000000,0,N/A,N/A,False,1
3,D,knn:{'n_neighbors': 1},0.996296,2.738129,0.983333,0.749242,2967,0.0256,"[2.05%, 3.19%]",True,2
4,D_strict,knn:{'n_neighbors': 1},NaN,NaN,NaN,0.126389,91,1.0,"[95.95%, 100.00%]",True,3


Validación interna del protocolo A (candidatos):


,candidate,val_class_accuracy,val_ece,val_end_to_end,automation_enabled,val_coverage,val_error_among_act,k
0,logreg:{'C': 0.1},0.9892,0.0256,0.9136,True,0.7186,0.0174,2
1,logreg:{'C': 1.0},0.9900,0.0084,0.9142,True,0.7186,0.0174,2
2,logreg:{'C': 10.0},0.9892,0.0044,0.9144,True,0.7192,0.0174,2
3,svm_rbf:{'C': 10.0},0.9844,0.5864,0.9103,True,0.8158,0.0194,2
4,knn:{'n_neighbors': 1},0.9714,0.0286,0.9008,True,0.7186,0.0197,2
5,random_forest:{'n_estimators': 300},0.9672,0.4206,0.8947,True,0.6933,0.0176,2


...

,candidate,val_class_accuracy,val_ece,val_end_to_end,automation_enabled,val_coverage,val_error_among_act,k
0,logreg:{'C': 0.1},0.8775,0.0961,0.0,False,0.0,None,1
1,logreg:{'C': 1.0},0.8750,0.0347,0.0,False,0.0,None,1
2,logreg:{'C': 10.0},0.8600,0.0427,0.0,False,0.0,None,1
3,svm_rbf:{'C': 10.0},0.7733,0.6564,0.0,False,0.0,None,1
4,knn:{'n_neighbors': 1},0.7633,0.2367,0.0,False,0.0,None,1
5,random_forest:{'n_estimators': 300},0.8008,0.4694,0.0,False,0.0,None,1


Validación interna del protocolo D (candidatos):


,candidate,val_class_accuracy,val_ece,val_end_to_end,automation_enabled,val_coverage,val_error_among_act,k
0,logreg:{'C': 0.1},0.9896,0.0258,0.9028,True,0.3781,0.0154,2
1,logreg:{'C': 1.0},0.9899,0.0093,0.9031,True,0.3781,0.0154,2
2,logreg:{'C': 10.0},0.9899,0.0040,0.9028,True,0.3786,0.0154,2
3,svm_rbf:{'C': 10.0},0.9826,0.5297,0.8976,True,0.3789,0.0154,2
4,knn:{'n_neighbors': 1},0.9701,0.0299,0.8882,True,0.3794,0.0168,2
5,random_forest:{'n_estimators': 300},0.9649,0.3982,0.8830,True,0.3794,0.0168,2


Validación interna del protocolo D_strict (candidatos):


,candidate,val_class_accuracy,val_ece,val_end_to_end,automation_enabled,val_coverage,val_error_among_act,k
0,logreg:{'C': 0.1},1.0000,0.0090,0.9679,True,0.7371,0.0132,3
1,logreg:{'C': 1.0},1.0000,0.0017,0.9679,True,0.7386,0.0132,3
2,logreg:{'C': 10.0},1.0000,0.0005,0.9679,True,0.7389,0.0132,3
3,svm_rbf:{'C': 10.0},0.9998,0.3018,0.9679,True,0.7378,0.0132,3
4,knn:{'n_neighbors': 1},0.9962,0.0038,0.9649,True,0.7399,0.0143,3
5,random_forest:{'n_estimators': 300},0.9986,0.2950,0.9665,True,0.7364,0.0128,3


**Calibración (solo validación):** la columna `val_ece` muestra que la logística está bien calibrada en todos los
protocolos, mientras que SVM (Platt multiclase) y Random Forest tienen ECE alto: sus probabilidades crudas no
sirven como confianza. En A la selección anidada puede preferir otro candidato si cubre más en validación; la
tabla deja ver ese trade-off.

## 5. Configuración desplegada (receta congelada)

Receta `classic-logreg-v1`: preprocesado + aumentación + PCA(150) + logística C=1 + el mismo procedimiento de
k/umbrales. **Salvedad:** familia y C se fijaron con la exploración inicial, que promedió validaciones de A–D; como
los cuatro protocolos cubren conjuntamente todas las imágenes, esa elección tocó imágenes que son test de otros
protocolos. Por eso estas cifras **no** se presentan como independientes; se comparan con las anidadas (arriba),
con la línea base y con la misma configuración sobre embeddings ResNet18 (sin aumentación).

In [8]:
hrows = []
for p, row in RS.items():
    for key, lab in (("nested_classic", "Selección anidada (independiente)"), ("recipe_classic", "Config. desplegada (clásica)"),
                     ("baseline_raw_1nn", "Base píxeles 1-NN (siempre ACT)"), ("recipe_resnet18", "Config. desplegada con ResNet18")):
        if row.get(key): hrows.append(rowfmt(p, lab, row[key]))
display(pd.DataFrame(hrows))
thr = pd.DataFrame({p: {**R["results"]["classic"][p]["thresholds"], "k": R["results"]["classic"][p]["k_selected_on_val"],
                        "val n ACT": R["results"]["classic"][p]["threshold_validation"]["val_n_act"],
                        "val error|ACT": R["results"]["classic"][p]["threshold_validation"]["val_error_among_act"],
                        "val desconocidos→ACT": R["results"]["classic"][p]["threshold_validation"]["unknown_false_accept_val"]["rate"]}
                    for p in RS}).T
print("Umbrales de la configuración desplegada por protocolo (elegidos en validación):"); display(thr)
fig = va.recipe_bars(RS); va.savefig(fig, "03_per_protocol"); plt.show()

,protocolo,modelo,exactitud,MAE°,éxito e2e,% ACT,n ACT,error|ACT,IC95,automatización
0,A,Selección anidada (independiente),1.000000,2.911346,0.985000,0.903056,3251.0,0.0009,"[0.03%, 0.27%]",True
1,A,Config. desplegada (clásica),1.000000,2.911346,0.985000,0.902778,3250.0,0.0009,"[0.03%, 0.27%]",True
2,A,Base píxeles 1-NN (siempre ACT),0.999722,5.928036,0.975833,1.000000,NaN,0.0242,"[1.96%, 2.97%]",—
3,A,Config. desplegada con ResNet18,1.000000,1.982566,0.987778,0.000000,0.0,N/A,N/A,False
4,B,Selección anidada (independiente),0.961667,65.515598,0.285000,0.000000,0.0,N/A,N/A,False
5,B,Config. desplegada (clásica),0.961667,65.515598,0.285000,0.000000,0.0,N/A,N/A,False
6,B,Base píxeles 1-NN (siempre ACT),0.929167,61.224215,0.285833,1.000000,NaN,0.7142,"[68.80%, 73.90%]",—
7,B,Config. desplegada con ResNet18,0.996667,48.323579,0.279167,0.000000,0.0,N/A,N/A,False
8,C,Selección anidada (independiente),0.987833,13.006580,0.697167,0.000000,0.0,N/A,N/A,False
9,C,Config. desplegada (clásica),0.988333,13.007589,0.697000,0.000000,0.0,N/A,N/A,False


Umbrales de la configuración desplegada por protocolo (elegidos en validación):


,min_class_confidence,max_angle_uncertainty_deg,unknown_distance,k,val n ACT,val error|ACT,val desconocidos→ACT
A,0.40,15.0,38.061432,2.0,2587.0,0.017395,NaN
B,0.80,-1.0,39.734272,1.0,0.0,NaN,NaN
C,0.90,-1.0,43.859887,1.0,0.0,NaN,NaN
D,0.90,12.5,24.424469,2.0,1361.0,0.015430,0.0
D_strict,0.95,30.0,23.084545,3.0,5318.0,0.013163,0.0


`error|ACT` = P(clase incorrecta, desconocido aceptado o |error| > 10° | el robot actúa), con IC de Wilson 95 %.
La línea base siempre actúa: en D sus aciertos sobre desconocidos son imposibles y cuentan como error. "N/A" =
política efectiva sin ninguna acción (automatización desactivada).

### 5.1 Robustez a perturbaciones

El modelo ya ajustado (mismos umbrales) se re-evalúa sobre el test perturbado de A (conocidos) y D estricto
(desconocidos): ruido gaussiano σ=2/4/8, brillo ×0.7/×1.3 y blur gaussiano σ=1 (semillas fijas). σ=8 y ×0.7/×1.3
están **fuera** del rango de aumentación (σ≤6, ×0.8–1.2): miden extrapolación.

In [9]:
for p in ("A", "D_strict"):
    print(f"Robustez — protocolo {p}")
    display(pd.DataFrame(va.robustness_table(R["results"]["classic"][p]["robustness"])).round(4))

Robustez — protocolo A


,perturbación,exactitud,MAE°,éxito e2e,% ACT,n ACT,error|ACT,IC95,conocidos rechazados
0,limpia,1.0000,2.9113,0.9850,0.9028,3250,0.0009,"[0.000314, 0.002711]",0.0003
1,ruido σ=2,1.0000,3.1489,0.9761,0.9028,3250,0.0015,"[0.000657, 0.003597]",0.0000
2,ruido σ=4,1.0000,3.9650,0.9581,0.8981,3233,0.0111,"[0.008054, 0.015377]",0.0003
3,ruido σ=8,0.9997,6.1179,0.9189,0.8550,3078,0.0240,"[0.019194, 0.030075]",0.0008
4,brillo ×0.7,0.9994,7.6661,0.9169,0.8061,2902,0.0148,"[0.011019, 0.019898]",0.0064
5,brillo ×1.3,1.0000,5.4990,0.9294,0.8650,3114,0.0167,"[0.012757, 0.021831]",0.0017
6,blur σ=1,1.0000,3.2386,0.9694,0.8997,3239,0.0049,"[0.003043, 0.00801]",0.0000


Robustez — protocolo D_strict


,perturbación,exactitud,MAE°,éxito e2e,% ACT,n ACT,error|ACT,IC95,conocidos rechazados,desconocidos a ACT,IC95 desconocidos a ACT,desconocidos rechazados
0,limpia,None,None,None,0.0639,46,1.0,"[0.922924, 1.0]",None,0.0639,"[0.048238, 0.084169]",0.7125
1,ruido σ=2,None,None,None,0.0625,45,1.0,"[0.921346, 1.0]",None,0.0625,"[0.047035, 0.082608]",0.7097
2,ruido σ=4,None,None,None,0.0611,44,1.0,"[0.919702, 1.0]",None,0.0611,"[0.045835, 0.081045]",0.7208
3,ruido σ=8,None,None,None,0.0597,43,1.0,"[0.917987, 1.0]",None,0.0597,"[0.044638, 0.07948]",0.7556
4,brillo ×0.7,None,None,None,0.0611,44,1.0,"[0.919702, 1.0]",None,0.0611,"[0.045835, 0.081045]",0.8292
5,brillo ×1.3,None,None,None,0.0556,40,1.0,"[0.912375, 1.0]",None,0.0556,"[0.041061, 0.074768]",0.7056
6,blur σ=1,None,None,None,0.0583,42,1.0,"[0.916198, 1.0]",None,0.0583,"[0.043443, 0.077912]",0.7444


**Lectura honesta.** Con el preprocesado y la aumentación, el ruido de sensor moderado (σ≤4) y el blur leve
apenas cambian la exactitud y la fracción automatizada. Las perturbaciones fuera del rango entrenado (σ=8, brillo
×0.7/×1.3) degradan el ángulo (MAE mayor) y la cobertura; el riesgo entre ACT observado sube y puede superar el
objetivo de 2 % (ver IC). La tasa de desconocidos que llegan a ACT se mantiene en el mismo orden bajo
perturbación, pero **no** es cero: los casi duplicados de un conocido pasan igual. En planta, cambios de
iluminación mayores a ±20 % exigen recalibrar o ampliar la aumentación, y la compuerta de calidad debe vigilar
brillo y nitidez.

### 5.2 Ángulo: error circular y elección de k en validación

In [10]:
outs = {p: va.load_outputs("recipe_classic", p, OUT_DIR) for p in ("A", "B", "C", "D")}
fig = va.angle_error_hist({p: outs[p] for p in "ABC"}); va.savefig(fig, "04_angle_error_hist"); plt.show()
kt = pd.DataFrame([{"protocolo": p, **r} for p in ("A", "B", "C", "D", "D_strict") for r in R["results"]["classic"][p]["k_table_val"]])
display(kt.pivot_table(index="k", columns="protocolo", values=["val_within_10", "val_mae_deg"]).round(3))
print("k elegido en validación:", {p: R["results"]["classic"][p]["k_selected_on_val"] for p in RS})

val_mae_deg                                  val_within_10                            
protocolo           A       B       C       D D_strict             A      B    C      D D_strict
k                                                                                               
1              16.201  61.902  53.086  17.275    7.224         0.790  0.271  0.0  0.777    0.964
2               9.866  62.907  53.086  11.056    3.992         0.923  0.159  0.0  0.912    0.966
3              10.887  63.929  53.086  12.118    4.387         0.918  0.153  0.0  0.906    0.968

k elegido en validación: {'A': 2, 'B': 1, 'C': 1, 'D': 2, 'D_strict': 3}


**Lectura.** Métrica: error circular `min(|a−b|, 360−|a−b|)`. La tabla de k es de **validación**: interpolar con
2–3 vecinos ayuda cuando el query cae entre referencias cercanas (A, D) y no con brechas grandes (B, C), donde
k=1 queda igual o mejor; por eso k se elige por protocolo en validación. En **B** (arco ciego) ningún método de
apariencia reconstruye las vistas no vistas; en **C** (12 vistas) el ángulo no alcanza la confiabilidad exigida.

### 5.3 Confianza: calibración, riesgo–cobertura y desconocidos

In [11]:
fig = va.reliability_and_risk(outs); va.savefig(fig, "05_reliability_risk"); plt.show()
fig = va.unknown_roc(outs["D"]); va.savefig(fig, "06_unknown_roc"); plt.show()
for p in ("D", "D_strict"):
    d = R["results"]["classic"][p]["decision"]
    print(p, {k: (round(v["rate"], 4), [round(x, 4) for x in v["wilson95"]]) for k, v in d.items() if k in ("unknown_rejected", "unknown_reaches_act")},
          "| criterio en validación:", R["results"]["classic"][p]["threshold_validation"]["unknown_false_accept_val"])

D {'unknown_rejected': (np.float64(0.6792), [0.6442, 0.7122]), 'unknown_reaches_act': (np.float64(0.0306), [0.0203, 0.0458])} | criterio en validación: {'k': 0, 'n': 720, 'rate': 0.0, 'wilson95': [0.0, 0.005307238489746928], 'max_allowed': 0.05, 'min_cases': 100, 'ok': True}
D_strict {'unknown_rejected': (np.float64(0.7125), [0.6784, 0.7444]), 'unknown_reaches_act': (np.float64(0.0639), [0.0482, 0.0842])} | criterio en validación: {'k': 0, 'n': 1440, 'rate': 0.0, 'wilson95': [0.0, 0.002660679675665253], 'max_allowed': 0.05, 'min_cases': 100, 'ok': True}


(1) La logística está bien calibrada (diagrama de confiabilidad). (2) En riesgo–cobertura, priorizar mayor
confianza o menor incertidumbre deja el riesgo de los primeros tramos por debajo del global; las curvas **no**
son estrictamente monótonas y en B no hay tramo con riesgo aceptable. (3) Criterio explícito de desconocidos:
≤ 5 % de pseudo-desconocidos de validación pueden llegar a ACT (con ≥ 100 vistas); el valor observado en test
(D estricto) se reporta con IC. Los desconocidos que llegan a ACT son casi siempre casi duplicados de un conocido
(p. ej. obj61 ≈ obj92).

### 5.4 Umbral de incertidumbre vs. tolerancia ±10°

El umbral (p. ej. 30°) es un corte sobre un score heurístico, no "error ≤ 30°". Se elige como el mayor umbral cuyo
riesgo observado en validación `P(clase incorrecta o |error| > 10° | ACT)` es ≤ 2 % con ≥ 50 ACT. La figura muestra
ese riesgo con IC de Wilson en validación (selección) y en test (descriptivo). Lo que se observa es que, una vez
aplicados la ambigüedad rotacional y la confianza, subir el corte por encima de ~10–15° añade pocos ACT y cambia
poco el riesgo observado en A; en D el riesgo observado crece con el corte. No es una garantía fuera de estos datos.

In [12]:
fig = va.sweep_figure(R["results"]["classic"], ("A", "D")); va.savefig(fig, "09_uncertainty_sweep"); plt.show()

## 6. Análisis de errores (configuración desplegada)

In [13]:
fails = va.per_object_failures({p: outs[p] for p in "ABC"}, top=8)
for p, rows in fails.items():
    print(f"Protocolo {p}: objetos con más fallos e2e"); display(pd.DataFrame(rows))
for p in ("B", "C"):
    fig = va.failure_examples_figure(ds, outs, p); va.savefig(fig, f"07_failures_{p}"); plt.show()
fA = outs["A"]; okA = (fA["label"] == fA["true_label"]) & (circular_error(fA["angle"], fA["true_angle"]) <= 10)
actA = fA["decision"] == "ACT"
print("A: fallos e2e totales:", int((~okA).sum()), "| fallos que llegaron a ACT:", int((~okA & actA).sum()))

Protocolo A: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj95,11,36,0,None
1,obj94,10,36,0,None
2,obj98,9,36,0,None
3,obj47,6,36,0,None
4,obj84,6,36,0,None
5,obj67,5,36,0,None
6,obj35,2,36,0,None
7,obj11,1,36,0,None


Protocolo B: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj98,12,12,0,NaN
1,obj1,11,12,0,NaN
2,obj96,11,12,0,NaN
3,obj84,11,12,0,NaN
4,obj66,11,12,0,NaN
5,obj79,11,12,0,NaN
6,obj100,10,12,8,obj77
7,obj94,10,12,0,NaN


Protocolo C: objetos con más fallos e2e


,object,failures,n,class_errors,confused_with
0,obj47,36,60,0,NaN
1,obj95,32,60,0,NaN
2,obj94,32,60,0,NaN
3,obj84,30,60,11,obj51
4,obj50,25,60,0,NaN
5,obj67,25,60,6,obj97
6,obj98,24,60,2,obj82
7,obj35,24,60,0,NaN


A: fallos e2e totales: 54 | fallos que llegaron a ACT: 3


**Lectura.** Los fallos angulares se concentran en pocos objetos: donas y taza (obj95, obj94, obj47; índice de
perfil alto) y vistas "gemelas" a 180° (obj98, obj67, obj84; índice LOO alto). El error en esos casos es de ±180°
o arbitrario; la mayoría se envía a REVIEW. Los errores de clase se dan entre objetos de la misma familia visual.

## 7. Costo: latencia y tamaño del artefacto

In [14]:
import subprocess
VERSION = "vision-1.0.1"
# --force: regenera la MISMA versión aún no publicada desde este notebook (las versiones publicadas no se tocan)
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "train_vision.py"), "--force"], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1500:]); print(r.stderr[-800:] if r.returncode else "")
from src.vision.inference import VisionPredictor
pred = VisionPredictor.load(ROOT / "models" / VERSION)
rng = np.random.default_rng(42); sample = ds.images[rng.choice(len(ds.images), 100, replace=False)]
lat = {"classic (pipeline completo, 1 imagen)": va.measure_latency(pred, sample)}
lat["baseline píxeles 1-NN (media/imagen en lote)"] = {"mean_ms": R["baselines"]["A"]["mean_ms_per_image_in_batch"]}
rl = va.measure_resnet_latency(sample, n=50)
lat["resnet18 CPU (solo embedding, 1 imagen)"] = rl["cpu"]
if "mps" in rl: lat["resnet18 MPS (solo embedding, 1 imagen)"] = rl["mps"]
display(pd.DataFrame(lat).T.round(2))
size_mb = sum(f.stat().st_size for f in (ROOT / "models" / VERSION).glob("*") if f.is_file()) / 1e6
print(f"Artefacto clásico (incluye code_snapshot): {size_mb:.1f} MB | pesos ResNet18 fp32: {rl['weights_mb_fp32']} MB (+ torch)")
fig = va.latency_figure(lat); va.savefig(fig, "08_latency"); plt.show()
COST = {"latency": lat, "classic_artifact_mb": size_mb, "resnet18_weights_mb": rl["weights_mb_fp32"]}

Receta congelada: classic-logreg-v1 (logreg {'C': 1.0}, PCA 150)
k (validación): 3 | clases con ángulo poco fiable: ['obj35', 'obj47', 'obj94', 'obj95', 'obj98']
Umbrales (validación): {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.83979578971863} {'thresholds': {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.83979578971863}, 'val_n_act': 5693, 'val_n_errors': 103, 'val_coverage': 0.7188131313131313, 'val_error_among_act': 0.01809239416827683, 'val_error_wilson95': [0.014940975631782035, 0.021893750473293453], 'target_error': 0.02, 'min_act': 50, 'unknown_false_accept_val': {'k': 59, 'n': 1440, 'rate': 0.04097222222222222, 'wilson95': [0.031896021617289036, 0.052491074584953654], 'max_allowed': 0.05, 'min_cases': 100, 'ok': True}, 'angle_threshold_source': 'validation', 'automation_enabled': True}
Entrenamiento final: 2.3s; clases simétricas: ['obj35', 'obj47', 'obj67', 'obj84', 'obj94', 'obj95', 'obj98'

,p50_ms,p95_ms,mean_ms
"classic (pipeline completo, 1 imagen)",18.02,24.61,16.82
baseline píxeles 1-NN (media/imagen en lote),NaN,NaN,0.30
"resnet18 CPU (solo embedding, 1 imagen)",26.97,31.20,NaN
"resnet18 MPS (solo embedding, 1 imagen)",18.79,30.65,NaN


Artefacto clásico (incluye code_snapshot): 17.6 MB | pesos ResNet18 fp32: 44.7 MB (+ torch)


**¿Justifica la brecha el costo?** Con la misma configuración, ResNet18 (sin aumentación) clasifica mejor en B/C,
pero tampoco habilita automatizar B ni C y en su validación no alcanza el criterio de riesgo en varios
protocolos; solo su embedding ya cuesta más en CPU que el pipeline clásico completo y añade ~45 MB de pesos más
PyTorch. **Producción = clásico**; ResNet18 queda como referencia.

## 8. Exportación del artefacto y registro

In [15]:
results = {"data_version_hash": DATA_HASH, "seed": 42, "feature_config": data["feature_cfg"].to_dict()}
results["recipe"] = vexp.to_jsonable(R)
results["recipe_summary"] = vexp.to_jsonable(RS)
results["cost"] = COST
results["symmetry"] = {"profile_score": {f"obj{k}": v for k, v in sym["profile_score"].items()},
                       "loo_score": {f"obj{k}": v for k, v in sym["loo_score"].items()}}
results["confusable_pairs"] = pairs
results["error_analysis"] = fails
results["protocols"] = {"D": {"classic": {"unknown_objects": [f"obj{o}" for o in spD["unknown_objects"]]}}}
p = va.write_results(vexp.to_jsonable(results)); print("escrito", p)
json.loads(p.read_text())  # JSON estricto (sin NaN)
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "train_vision.py"), "--force"], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1200:])
meta = json.loads((ROOT / "models" / VERSION / "metadata.json").read_text())
print({k: meta[k] for k in ("version", "classifier", "k_angle_refine", "thresholds", "symmetric_classes", "artifact_size_mb",
                            "model_sha256", "code_snapshot_sha256", "git")})
print(f"Tiempo total del notebook: {(time.time() - T0) / 60:.1f} min")

escrito /Users/nico/Documents/IA/docs/report/results.json


Receta congelada: classic-logreg-v1 (logreg {'C': 1.0}, PCA 150)
k (validación): 3 | clases con ángulo poco fiable: ['obj35', 'obj47', 'obj94', 'obj95', 'obj98']
Umbrales (validación): {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.83979578971863} {'thresholds': {'min_class_confidence': 0.99, 'max_angle_uncertainty_deg': 30.0, 'unknown_distance': 19.83979578971863}, 'val_n_act': 5693, 'val_n_errors': 103, 'val_coverage': 0.7188131313131313, 'val_error_among_act': 0.01809239416827683, 'val_error_wilson95': [0.014940975631782035, 0.021893750473293453], 'target_error': 0.02, 'min_act': 50, 'unknown_false_accept_val': {'k': 59, 'n': 1440, 'rate': 0.04097222222222222, 'wilson95': [0.031896021617289036, 0.052491074584953654], 'max_allowed': 0.05, 'min_cases': 100, 'ok': True}, 'angle_threshold_source': 'validation', 'automation_enabled': True}
Entrenamiento final: 2.2s; clases simétricas: ['obj35', 'obj47', 'obj67', 'obj84', 'obj94', 'obj95', 'obj98'

## 9. Conclusiones

* **Cifras independientes** = selección anidada por protocolo (sección 4). La configuración desplegada
  (`vision-1.0.1`, logística C=1) se reporta aparte con su salvedad de selección.
* **Robustez:** el preprocesado (mediana + supresión de fondo) y la aumentación eliminan la fragilidad ante ruido
  de sensor moderado; perturbaciones fuera del rango entrenado siguen degradando ángulo y cobertura (sección 5.1).
* **Protocolos:** A y D son automatizables con riesgo observado bajo (IC en tablas); B y C no: la política efectiva
  es 0 ACT (todo a REVIEW). Para dar de alta un SKU: ≥ 12 vistas cubriendo ≥ 300° para staging y referencias densas
  (cada 5–10°) para habilitar ACT.
* **Limitaciones:** dataset de laboratorio; desconocidos casi duplicados pueden llegar a ACT; el score angular es
  heurístico. Ninguna regla de decisión garantiza acierto: el riesgo es el observado en estos datos.